# Capítulo 5: Gradiente Descendente

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 8 de Grus (2019).

> Quem se vangloria da própria descendência está se vangloriando daquilo que deve a outros.
>
> — Sêneca

Boa parte da ciência de dados é resolver um problema de otimização disfarçado. "Ajustar um modelo" quase sempre significa achar os parâmetros que minimizam algum erro ou maximizam alguma verossimilhança — e "melhor modelo", na prática, é o nome bonito que damos à solução desse problema. Este capítulo constrói, do zero, a técnica que este livro inteiro vai usar para resolver esses problemas: o **gradiente descendente**.

A ideia cabe numa frase: calcule a direção em que uma função cresce mais rápido, e ande na direção oposta, em passos pequenos, até quase não se mover mais. O capítulo constrói essa ideia em camadas — o que é o gradiente, como estimá-lo, como usá-lo para minimizar uma função, quanto andar a cada passo, como usar tudo isso para ajustar um modelo a dados, e como fazer isso em escala — e chega, já na seção 5.3, a uma função de **cinco linhas**, `gradient_step`: a peça que os capítulos seguintes de fato chamam.

> **❗ Importante — Por que este capítulo pesa mais que os outros**
>
> `gradient_step` não é só mais um algoritmo do livro. É a máquina que treina praticamente tudo o que vem depois dela. Os capítulos [11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html) (Regressão Linear Simples), [12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) (Regressão Múltipla), [13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) (Regressão Logística), [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) (Redes Neurais) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) (Deep Learning) chamam essa função — direta ou indiretamente — para ajustar os parâmetros dos modelos que constroem. Nenhum deles reimplementa a ideia; todos importam a mesma peça que você vai escrever aqui.
>
> Os modelos que **não** passam por este capítulo são os que não se ajustam descendo um gradiente, e vale saber quais são desde já: o k-vizinhos mais próximos ([Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html)), cuja previsão é uma busca nos dados guardados; o Naive Bayes ([Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html)), que se treina **contando**; as árvores de decisão ([Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html)), construídas por uma escolha gulosa repetida; e o k-means ([Capítulo 17](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html)), que alterna dois passos exatos.
>
> Isso não quer dizer que esses quatro não tenham nada a ajustar — o `k` do k-vizinhos e o do k-means são escolhidos de fora, por quem usa o modelo, em vez de aprendidos a partir dos dados. A diferença é essa: um parâmetro que o gradiente descobre, contra um que você decide.
>
> Isso inverte a ordem usual de aprender machine learning. Normalmente você vê um algoritmo, depois outro, e só bem mais tarde percebe que todos treinam do mesmo jeito por baixo. Aqui você constrói o "por baixo" primeiro. Depois deste capítulo, cinco capítulos inteiros deixam de ser caixas-pretas de treino e passam a ser "mais um jeito de calcular um gradiente e chamar `gradient_step`".

Ao final deste capítulo, você será capaz de:

- Explicar a ideia por trás do gradiente descendente: por que seguir a direção oposta ao gradiente reduz o valor de uma função
- Estimar a derivada e o gradiente de uma função por diferença finita, e explicar por que essa estimativa não é o que se usa na prática
- Implementar `gradient_step` e usá-lo para minimizar uma função simples a partir de um ponto aleatório
- Diagnosticar, numericamente, o que acontece quando o tamanho do passo é grande demais ou pequeno demais
- Ajustar um modelo linear a dados calculando o gradiente do erro quadrático médio e aplicando gradiente descendente
- Distinguir gradiente descendente em lote, minibatch e estocástico, e explicar o compromisso entre eles

## Seções

| Seção | Tópico |
|---|---|
| [5.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/01-a-ideia-por-tras-do-gradiente.html) | A Ideia por Trás do Gradiente Descendente |
| [5.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/02-estimando-o-gradiente.html) | Estimando o Gradiente |
| [5.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html) | Usando o Gradiente |
| [5.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html) | Escolhendo o Tamanho do Passo |
| [5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) | Ajustando Modelos com Gradiente Descendente |
| [5.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/06-minibatch-e-estocastico.html) | Minibatch e Gradiente Estocástico |

## A Ideia por Trás do Gradiente Descendente

> **📌 Nota**
>
> Esta seção corresponde a *The Idea Behind Gradient Descent*, do capítulo 8 de Grus (2019).

Em ciência de dados, resolver um problema frequentemente se resume a achar, para alguma função $f$, o vetor de entrada que a minimiza ou maximiza. Isso significa resolver vários problemas de otimização sem chamar um otimizador pronto. A técnica que vamos usar, o **gradiente descendente**, é justamente a que melhor se presta a isso: cabe em poucas linhas e não esconde nada.

Suponha que temos alguma função $f$ que recebe como entrada um vetor de números reais e devolve um único número real. Uma função simples desse tipo é a que já apareceu no [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html):

In [ ]:
from scratch.linear_algebra import Vector, dot

def sum_of_squares(v: Vector) -> float:
    """Calcula a soma dos elementos ao quadrado em v"""
    return dot(v, v)

sum_of_squares([1, 2, 3])

Frequentemente vamos precisar maximizar ou minimizar funções como essa — isto é, achar o `v` de entrada que produz o maior (ou o menor) valor possível.

> **🔷 Conceito**
>
> Para funções como a nossa, o **gradiente** — se você lembra do seu cálculo, é o vetor de derivadas parciais — aponta, no espaço de entrada, a direção em que a função cresce mais rápido.
>
> Uma forma de maximizar uma função é, então: escolher um ponto de partida aleatório, calcular o gradiente, dar um passo pequeno na direção do gradiente (a direção que faz a função crescer mais) e repetir a partir do novo ponto. Da mesma forma, dá para *minimizar* uma função dando passos pequenos na direção **oposta**.

### O procedimento

O procedimento inteiro é este:

1. Escolha um ponto de partida qualquer.
2. Calcule o gradiente nesse ponto.
3. Dê um passo pequeno na direção oposta ao gradiente (se o objetivo é minimizar).
4. Repita a partir do novo ponto.

In [ ]:
# Figura: Descida de gradiente sobre f(x, y) = x² + 5y²: as curvas de nível são elipses, e o caminho se curva para ficar perpendicular a cada uma delas
import matplotlib.pyplot as plt

def f(x: float, y: float) -> float:
    return x ** 2 + 5 * y ** 2

xs = [i / 10 for i in range(-60, 61)]
ys = [i / 10 for i in range(-60, 61)]
Z = [[f(x, y) for x in xs] for y in ys]

# um ponto de partida qualquer, e 15 passos manuais na direção oposta ao
# gradiente — a mesma conta que a seção 5.3 vai formalizar em gradient_step
x, y = -5.0, 4.0
caminho = [(x, y)]
for _ in range(15):
    grad_x, grad_y = 2 * x, 10 * y
    x, y = x - 0.05 * grad_x, y - 0.05 * grad_y
    caminho.append((x, y))

fig, ax = plt.subplots(figsize=(6, 5))
ax.contour(xs, ys, Z, levels=15, colors="0.65")
cam_x = [p[0] for p in caminho]
cam_y = [p[1] for p in caminho]
ax.plot(cam_x, cam_y, "o-", color="#c00000")
ax.annotate("início", (cam_x[0], cam_y[0]), textcoords="offset points",
            xytext=(8, 6), fontsize=9, color="#c00000")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

As curvas de nível do gráfico são as curvas onde `f` vale o mesmo — elipses, não círculos: como o termo em `y` pesa cinco vezes mais que o termo em `x`, para manter `f` constante uma variação em `y` precisa ser bem menor que a variação equivalente em `x`. Isso faz a tigela ser mais íngreme na direção de `y` do que na de `x`.

Repare no que isso faz com o caminho. Cada ponto vermelho ainda se move perpendicular à curva de nível em que está — essa regra não muda —, mas agora "perpendicular" não aponta para a origem: o passo inicial desce quase reto (a direção íngreme domina), e só depois de `y` encolher é que o caminho vira e passa a rastejar ao longo de `x`, a direção rasa. O caminho **se curva**. Numa tigela circular — o caso hipotético em que `x` e `y` pesassem igual, sem um termo dominando o outro —, isso seria invisível: direção radial e direção perpendicular à curva de nível coincidiriam por acaso. Aqui, não: a curvatura da função aparece no formato do caminho, e vai reaparecer, com números, quando a [seção 5.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html) explicar por que o tamanho certo do passo depende dela.

> **⚠️ Atenção**
>
> Uma ressalva do próprio Grus (2019): se uma função tem um único mínimo global, este procedimento provavelmente o encontra. Se a função tem múltiplos mínimos *locais*, o procedimento pode "achar" o mínimo errado — nesse caso, você pode rodar o procedimento de novo a partir de pontos de partida diferentes. E se a função não tiver mínimo nenhum, é possível que o procedimento rode para sempre.

Falta uma peça: como, exatamente, calculamos o gradiente de uma função? É isso que a próxima seção resolve.

## Estimando o Gradiente

> **📌 Nota**
>
> Esta seção corresponde a *Estimating the Gradient*, do capítulo 8 de Grus (2019).

Se $f$ é uma função de uma variável, sua derivada num ponto $x$ mede o quanto $f(x)$ muda quando fazemos uma variação muito pequena em $x$. Ela é definida como o limite dos quocientes de diferença:

$$f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}$$

Em código:

In [ ]:
from typing import Callable

def difference_quotient(f: Callable[[float], float],
                        x: float,
                        h: float) -> float:
    return (f(x + h) - f(x)) / h

Não vamos precisar da definição formal de limite: aqui, "limite" é o valor do qual a expressão se aproxima conforme `h` encolhe.

A derivada é a inclinação da reta tangente em $(x, f(x))$, enquanto o quociente de diferença é a inclinação da reta secante que passa por $(x, f(x))$ e $(x+h, f(x+h))$ — a reta "quase tangente". Conforme `h` fica menor, a reta secante se aproxima cada vez mais da tangente.

Para muitas funções é fácil calcular a derivada exatamente. Por exemplo, a função `square`:

In [ ]:
def square(x: float) -> float:
    return x * x

def derivative(x: float) -> float:
    return 2 * x

tem derivada `2 * x`, o que é fácil de conferir calculando explicitamente o quociente de diferença e tomando o limite.

### Quando calcular a derivada não é uma opção

E se você não pudesse (ou não quisesse) achar a derivada? Embora não dê para tomar limites em Python, dá para *estimar* derivadas avaliando o quociente de diferença para um `h` bem pequeno:

In [ ]:
difference_quotient(square, 3, h=0.001), derivative(3)

A estimativa fica bem perto do valor exato. Mas "perto" depende inteiramente da escolha de `h` — e existe um jeito errado de escolher `h`, que vale a pena ver antes de seguir em frente.

Para `square`, $\frac{(x+h)^2 - x^2}{h} = 2x + h$: matematicamente, a estimativa erra por exatamente `h`, não importa o `x`. Quanto menor o `h`, menor o erro — em teoria, sem limite. Na prática, `h` é um número de ponto flutuante, e ponto flutuante tem precisão finita. Se `h` for pequeno demais *em relação a* `x`, a soma `x + h` perde dígitos ao ser arredondada para o `float` mais próximo — no limite, `x + h` arredonda de volta para o próprio `x`, `f(x+h) - f(x)` vira exatamente `0`, e a estimativa desaba para `0`, não importa quão perto do zero `h` esteja.

In [ ]:
# Figura: Erro da estimativa por diferença finita em função de h, para square(x) = x² em x = 3 (escala log-log)
x0 = 3.0
exato = derivative(x0)

hs = [10 ** (-k / 4) for k in range(65)]  # h de 1 a 1e-16, log-espaçado
erros = [abs(difference_quotient(square, x0, h) - exato) for h in hs]

from matplotlib import pyplot as plt

plt.plot(hs, erros, 'o-', markersize=3)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('h')
plt.ylabel('erro |estimativa − derivada exata|')
plt.show()

O gráfico tem forma de U. Do lado direito (`h` grande), o erro cai junto com `h` numa reta limpa — é o erro de truncamento que acabamos de derivar, `erro ≈ h`, e a inclinação 1 em escala log-log é exatamente essa relação. Isso continua até `h` chegar perto de `10⁻⁸`, onde o erro atinge um vale. Do lado esquerdo desse vale, encolher `h` **piora** a estimativa — mas não numa curva lisa: o erro sobe aos trancos, com picos e quedas, porque só existe um punhado de valores de `float` distintos entre `x` e `x + h` nessa faixa, e qual deles a soma arredonda para depende de detalhes de bit que mudam de `h` para `h`. O que é liso é a tendência: o piso do serrilhado sobe, até bater no teto — um erro de `6`, ou seja, uma estimativa de `0` — quando `h` fica menor do que a precisão de `double` consegue distinguir de `x`.

> **⚠️ Atenção — `h` pequeno demais não é mais preciso — é ruído**
>
> O ponto mais baixo do U não é `h → 0`. Fica por volta de `10⁻⁸`, perto de $\sqrt{\epsilon_{\text{máquina}}}$ (a raiz quadrada do épsilon de máquina de um `double`, cerca de `2,2 × 10⁻¹⁶`) — um resultado clássico de análise numérica, não uma peculiaridade deste exemplo. Antes desse ponto, quem domina é o erro de truncamento (`h` grande demais para a aproximação linear valer); depois dele, quem domina é o cancelamento catastrófico (`h` pequeno demais para `x + h` ser representado sem perder informação).
>
> Os valores de `h` usados neste capítulo — `0.001` na comparação acima, `0.0001` como padrão de `estimate_gradient`, logo adiante — não foram escolhidos por acaso: estão bem à direita do fundo do U, longe o bastante do cancelamento para sobrar margem, e ainda assim pequenos o bastante para o erro de truncamento ser desprezível.

### Derivadas parciais, em várias dimensões

Quando $f$ é uma função de muitas variáveis, ela tem múltiplas **derivadas parciais**, cada uma indicando como $f$ muda quando fazemos uma pequena mudança numa única variável de entrada.

Calculamos a $i$-ésima derivada parcial tratando $f$ como uma função de apenas sua $i$-ésima variável, mantendo as outras fixas:

In [ ]:
from scratch.linear_algebra import Vector

def partial_difference_quotient(f: Callable[[Vector], float],
                                v: Vector,
                                i: int,
                                h: float) -> float:
    """Retorna o i-ésimo quociente de diferença parcial de f em v"""
    w = [v_j + (h if j == i else 0)    # soma h só ao i-ésimo elemento de v
         for j, v_j in enumerate(v)]

    return (f(w) - f(v)) / h

depois do que podemos estimar o gradiente da mesma forma:

In [ ]:
def estimate_gradient(f: Callable[[Vector], float],
                      v: Vector,
                      h: float = 0.0001):
    return [partial_difference_quotient(f, v, i, h)
            for i in range(len(v))]

from scratch.linear_algebra import dot

def sum_of_squares(v: Vector) -> float:
    return dot(v, v)

estimate_gradient(sum_of_squares, [3.0, 4.0, 5.0])  # exato seria [6, 8, 10]

Uma cópia do código do livro-texto que mora dentro deste repositório (`scratch/gradient_descent.py`) também define `estimate_gradient`, mas lá `partial_difference_quotient` vive dentro de uma função de demonstração e não é visível fora dela — importar funciona, chamar estoura com `NameError`. Por isso escrevemos as duas por conta própria aqui; a [seção 5.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html) explica como o resto do capítulo reaproveita código entre seções.

Um problema sério dessa abordagem de "estimar usando quocientes de diferença" é que ela é computacionalmente cara. Se `v` tem tamanho `n`, `estimate_gradient` precisa avaliar `f` em `2n` entradas diferentes. Se você está estimando gradientes repetidamente, está fazendo bastante trabalho extra. Em tudo o que fizermos daqui para frente, vamos usar matemática para calcular nossas funções de gradiente explicitamente, em vez de estimá-las.

> **💡 Dica — Na prática: diferenciação automática**
>
> O `estimate_gradient` que você acabou de escrever tem um equivalente pronto:
>
> ```python
> from scipy.optimize import approx_fprime
>
> approx_fprime(v, f, epsilon=1e-4)
> ```
>
> Mesma ideia, mesma conta — quociente de diferença, uma dimensão de cada vez, com o mesmo compromisso do U que você acabou de ver: `epsilon` grande demais trunca, pequeno demais cancela.
>
> Mas sistemas de aprendizado de máquina reais quase nunca estimam gradientes assim. Eles usam **diferenciação automática** (*autodiff*): em vez de perturbar cada entrada e reavaliar a função do zero, o framework registra cada operação aritmética feita e aplica a regra da cadeia de trás para frente, obtendo o gradiente **exato** — não uma estimativa, e portanto sem o compromisso truncamento/cancelamento acima — a um custo próximo ao de uma única avaliação da função, não `2n` delas. É o que `torch.autograd` (PyTorch) e `jax.grad` (JAX) fazem, e é o mecanismo por trás da palavra *backpropagation*, que reaparece nos capítulos sobre [redes neurais](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [deep learning](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html).
>
> A diferença importa em escala: uma rede com um milhão de parâmetros levaria `estimate_gradient` a avaliar a função dois milhões de vezes a cada passo. Autodiff faz isso em uma passada.

## Usando o Gradiente

> **📌 Nota**
>
> Esta seção corresponde a *Using the Gradient*, do capítulo 8 de Grus (2019).

Derivando à mão, a derivada parcial de $\sum_j v_j^2$ em relação a $v_i$ é $2v_i$ — a soma dos quadrados só depende de $v_i$ através do termo $v_i^2$, e a derivada de $x^2$ é $2x$. Confira contra a seção anterior: `estimate_gradient(sum_of_squares, [3.0, 4.0, 5.0])` devolveu `[6.0001, 8.0001, 10.0001]` — o exato `[6, 8, 10]`, que é $[2 \cdot 3, 2 \cdot 4, 2 \cdot 5]$, mais o `h` que a seção 5.2 previu. Daqui em diante calculamos gradientes assim, com a fórmula fechada — a estimativa numérica da seção anterior passa a ser o instrumento de **conferência**, não o de produção.

É fácil ver, então, que a função `sum_of_squares` é mínima quando sua entrada `v` é um vetor de zeros. Mas imagine que não soubéssemos disso. Vamos usar gradientes para achar o mínimo entre todos os vetores tridimensionais. Vamos simplesmente escolher um ponto de partida aleatório e então dar passos pequenos na direção oposta ao gradiente, até chegar a um ponto em que o gradiente é muito pequeno:

In [ ]:
import random
from scratch.linear_algebra import Vector, distance, add, scalar_multiply

def gradient_step(v: Vector, gradient: Vector, step_size: float) -> Vector:
    """Anda `step_size` na direção do `gradient`, a partir de `v`"""
    assert len(v) == len(gradient)
    step = scalar_multiply(step_size, gradient)
    return add(v, step)

def sum_of_squares_gradient(v: Vector) -> Vector:
    return [2 * v_i for v_i in v]

`gradient_step` é a peça inteira do capítulo — as duas linhas de código que fazem o trabalho. Dado um ponto `v`, um gradiente naquele ponto e um `step_size` — positivo para andar *com* o gradiente, negativo para andar *contra* ele —, ela devolve o próximo ponto.

Repare que o sinal fica por conta de quem chama a função: é você que passa `-0.01` para descer. As bibliotecas de verdade (PyTorch, scikit-learn) escondem esse sinal — recebem um `learning_rate` **positivo** e embutem o `-1` por dentro. Se mais adiante você vir `lr=0.001` em código de produção, é o mesmo `step_size=-0.001` que vamos passar aqui, só que com o sinal de descida já aplicado por quem escreveu a biblioteca.

Agora usamos as duas funções para minimizar `sum_of_squares`, partindo de um ponto aleatório e dando 1.000 passos pequenos na direção oposta ao gradiente:

In [ ]:
random.seed(0)

# escolhe um ponto de partida aleatório
v = [random.uniform(-10, 10) for i in range(3)]
v

Aqui, *epoch* é só o nome que damos a cada iteração do laço — cada tentativa de melhorar `v` um pouco. A partir da [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html), quando houver um conjunto de dados de verdade para percorrer, a palavra ganha um sentido mais específico: uma passagem completa por ele.

In [ ]:
# Figura: Distância de v à origem ao longo de 1000 epochs (escala log)
distancias = [distance(v, [0, 0, 0])]

for epoch in range(1000):
    grad = sum_of_squares_gradient(v)    # calcula o gradiente em v
    v = gradient_step(v, grad, -0.01)    # dá um passo negativo na direção do gradiente
    distancias.append(distance(v, [0, 0, 0]))

assert distance(v, [0, 0, 0]) < 0.001    # v deve estar perto de 0

from matplotlib import pyplot as plt

plt.plot(range(len(distancias)), distancias)
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("distância de v até a origem")
plt.show()

In [ ]:
v, distance(v, [0, 0, 0])

A semente fixa esconde o que importa: troque-a e o ponto de partida muda, mas o destino não — `v` termina praticamente na origem em qualquer partida.

> **🔷 Conceito**
>
> A curva acima é (quase) uma reta porque o eixo vertical está em escala logarítmica — e isso não é acidente.
>
> Cada componente de `v` é multiplicado pelo mesmo fator a cada passo: `gradient_step` calcula `v + step_size * gradient`, e como `sum_of_squares_gradient(v)` é `2 * v`, isso é `v + (-0.01) * (2 * v)`, ou seja, `v * 0.98`. A distância à origem, portanto, decai geometricamente com razão `0.98` por epoch — depois de 1.000 epochs, `0.98 ** 1000`, um número na casa de `10⁻⁹`, multiplicado pela distância inicial. Uma reta em escala log **é** decaimento geométrico.
>
> Essa é uma propriedade de `sum_of_squares` em particular — a maioria das funções que vamos minimizar neste livro não tem uma forma fechada tão simples assim para a taxa de convergência. Mas o padrão qualitativo (queda rápida no início, cada vez mais lenta perto do mínimo) é típico do gradiente descendente com passo fixo, e vai reaparecer.

> **💡 Dica — Na prática: otimizadores prontos**
>
> O que você acabou de escrever — calcule o gradiente, ande na direção oposta, repita — é, no fundo, o que qualquer otimizador de propósito geral faz. A versão de biblioteca:
>
> ```python
> from scipy.optimize import minimize
>
> resultado = minimize(lambda v: sum(x**2 for x in v), x0=[6.9, 5.2, -1.6])
> resultado.x
> ```
>
> A diferença não é a ideia — é a sofisticação da escolha de passo e de direção. `scipy.optimize.minimize`, por padrão, usa BFGS: além do gradiente, ele mantém uma aproximação da curvatura da função (uma aproximação da matriz Hessiana) para decidir não só a direção, mas um tamanho de passo razoável a cada iteração, com busca em linha para não passar do ponto. `gradient_step`, com passo fixo, não sabe nada sobre curvatura — funciona porque `sum_of_squares` é uma tigela bem-comportada.
>
> A versão simples que você escreveu reaparece, praticamente sem alteração, dentro de todo otimizador que você vai encontrar nos capítulos [11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html), [12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html), [13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html), [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) deste livro. `gradient_step` não é uma versão de brinquedo de um otimizador de verdade — é o núcleo de um.

## Escolhendo o Tamanho do Passo

> **📌 Nota**
>
> Esta seção corresponde a *Choosing the Right Step Size*, do capítulo 8 de Grus (2019).

A lógica de andar contra o gradiente é clara. Quão longe andar não é. Na verdade, escolher o tamanho certo do passo é mais arte do que ciência. As opções populares incluem:

- Usar um tamanho de passo fixo.
- Encolher o tamanho do passo gradualmente, ao longo do tempo.
- A cada passo, escolher o tamanho de passo que minimiza o valor da função objetivo.

A última opção parece ótima, mas é, na prática, um cálculo caro — a cada passo, seria preciso resolver um novo problema de otimização só para decidir o tamanho do próximo passo. Para manter as coisas simples, este livro vai quase sempre usar um tamanho de passo fixo. E o tamanho de passo que "funciona" depende do problema: pequeno demais, e o gradiente descendente demora uma eternidade; grande demais, e você dá passos gigantescos que podem levar a função para fora do domínio — um overflow, ou um logaritmo de número negativo. Então, no fim, é preciso experimentar.

`gradient_step` e `sum_of_squares_gradient` já foram escritas na [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html). Em vez de reescrevê-las, importamos:

In [ ]:
from scratch.gradient_descent import gradient_step, sum_of_squares_gradient

> **📌 Nota — Por que importar, e não reescrever**
>
> Cada `.qmd` deste livro renderiza com seu próprio kernel Python — nomes definidos numa página não existem em outra. A seção 5.3 escreveu `gradient_step` e `sum_of_squares_gradient`; esta seção precisa das duas de novo, e importa em vez de reescrever. (Nem tudo em `scratch/gradient_descent.py` é seguro de importar assim — a seção 5.2 mostrou um caso em que não é.)

Vamos ver o que acontece minimizando `sum_of_squares` a partir do mesmo ponto de partida, com três tamanhos de passo diferentes:

In [ ]:
# Figura: Efeito do tamanho do passo: distância à origem para três valores de step_size, ao longo de 15 epochs (escala log)
import random
from scratch.linear_algebra import distance

def trajetoria_distancias(step_size: float, n_epochs: int):
    random.seed(0)
    v = [random.uniform(-10, 10) for i in range(3)]
    distancias = [distance(v, [0, 0, 0])]
    for epoch in range(n_epochs):
        grad = sum_of_squares_gradient(v)
        v = gradient_step(v, grad, step_size)
        distancias.append(distance(v, [0, 0, 0]))
    return distancias

certo = trajetoria_distancias(-0.01, 15)
grande = trajetoria_distancias(-1.5, 15)
pequeno = trajetoria_distancias(-0.00001, 15)

from matplotlib import pyplot as plt

epochs = range(16)
plt.plot(epochs, certo, label="step_size = -0.01 (certo)")
plt.plot(epochs, grande, label="step_size = -1.5 (grande demais)")
plt.plot(epochs, pequeno, label="step_size = -0.00001 (pequeno demais)")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("distância de v até a origem")
plt.legend()
plt.show()

In [ ]:
certo[0], certo[-1], grande[-1], pequeno[-1]

Os três partem exatamente do mesmo ponto, a `8.75` unidades da origem. Com `step_size = -0.01`, a distância cai para menos de `6.5` em 15 epochs — a mesma curva suave da seção anterior. Com `step_size = -0.00001`, ela mal se move: depois de 15 epochs ainda está acima de `8.74`.

Quanto tempo levaria esse passo pequeno demais para de fato chegar perto de zero? Dá para contar, em vez de estimar de cabeça:

In [ ]:
random.seed(0)
v_lento = [random.uniform(-10, 10) for i in range(3)]

n_epochs = 0
while distance(v_lento, [0, 0, 0]) >= 0.001:
    grad = sum_of_squares_gradient(v_lento)
    v_lento = gradient_step(v_lento, grad, -0.00001)
    n_epochs += 1

n_epochs

**453.845 epochs** — não "dezenas de milhares", *centenas* de milhares. Num capítulo cuja lição é sobre grandeza de passo e convergência, vale conferir a ordem de grandeza direito: essa distância, que o passo certo cruzou em 1.000 epochs, levaria mais de 450 vezes isso com o passo pequeno demais.

> **⚠️ Atenção — Passo grande demais diverge — e não devagar**
>
> Com `step_size = -1.5`, a distância **dobra a cada epoch**: `8.75`, `17.5`, `35`, `70`... e depois de 15 epochs já passa de `280.000`. Isso não é um valor extremo escolhido a dedo — é a mesma conta da caixa acima, só que com o sinal trocado.
>
> `gradient_step` multiplica `v` por `1 + 2 · step_size` a cada passo, porque `sum_of_squares_gradient(v)` é `2v`. Com `step_size = -0.01`, esse fator é `0.98`: encolhe. Com `step_size = -1.5`, o fator é `1 - 3 = -2`: cada passo não só deixa de se aproximar do mínimo, como **supercorrige para o lado oposto** e ainda cresce em magnitude. Um passo grande demais em relação à curvatura da função não converge devagar — ele diverge, e geometricamente.

A moral: o tamanho do passo certo depende de quão curva é a função que você está minimizando, e você raramente sabe isso de antemão. Por isso Grus (2019) aponta a segunda opção da lista — encolher o passo ao longo do tempo — como uma saída de compromisso: comece com um passo razoavelmente grande (convergência rápida no início) e vá encolhendo (evita a divergência quando já estiver perto do mínimo, onde a curvatura relativa importa mais).

> **🔷 Conceito**
>
> Não existe um `step_size` certo, universal — existe um certo *para uma curvatura*. Pequeno demais desperdiça iterações inteiras sem necessidade (a caixa de `453.845` epochs, acima); grande demais transforma cada passo numa supercorreção que cresce sem parar (a caixa de divergência geométrica). Os otimizadores que reaparecem no resto deste livro — e os que você vai encontrar em qualquer biblioteca — existem, em boa parte, para não deixar essa escolha inteiramente a cargo de quem treina o modelo.

> **💡 Dica — Na prática: o que se faz com isso**
>
> Nenhuma biblioteca séria de aprendizado de máquina usa um `step_size` fixo do jeito que este capítulo usa. O que existe, em vez disso, é uma família de **otimizadores adaptativos** — SGD com momentum, RMSProp, Adam — que ajustam o tamanho efetivo do passo automaticamente, com base no histórico dos gradientes recentes. A ideia comum a todos eles: acelerar quando os gradientes apontam consistentemente na mesma direção, e frear quando eles oscilam de sinal a cada passo — o mecanismo por trás da divergência geométrica da caixa acima (o fator `1 - 3 = -2` troca o sinal de `v` a cada epoch), ainda que o gráfico, que plota só a distância — sempre positiva —, não deixe essa troca de sinal visível, apenas o crescimento que ela produz.
>
> Frameworks de redes neurais também usam **cronogramas de taxa de aprendizado** (*learning rate schedules*): começar com um passo maior e reduzi-lo — em degraus, ou suavemente — conforme o treino avança, exatamente a segunda opção da lista desta seção. Você vai ver essas ideias de novo, com nome e implementação, nos capítulos sobre [redes neurais](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [deep learning](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html). O que muda ali não é o princípio — ainda é "ande contra o gradiente, em passos pequenos" — é só o quanto de trabalho extra o otimizador faz para escolher, a cada passo, o quão pequeno "pequeno" deveria ser.

## Ajustando Modelos com Gradiente Descendente

> **📌 Nota**
>
> Esta seção corresponde a *Using Gradient Descent to Fit Models*, do capítulo 8 de Grus (2019).

Neste livro, vamos usar gradiente descendente para ajustar modelos parametrizados a dados. No caso usual, temos um conjunto de dados e algum modelo hipotético para os dados, que depende — de forma diferenciável — de um ou mais parâmetros. Também temos uma função de **perda** (*loss*) que mede o quanto o modelo erra nos dados — quanto menor, melhor o ajuste.

Se pensarmos nos dados como fixos, nossa função de perda nos diz o quanto um conjunto específico de parâmetros é bom ou ruim. Isso significa que podemos usar gradiente descendente para achar os parâmetros do modelo que deixam a perda a menor possível. Vamos ver um exemplo simples:

In [ ]:
# x varia de -50 a 49, y é sempre 20 * x + 5
inputs = [(x, 20 * x + 5) for x in range(-50, 50)]

Neste caso *sabemos* os parâmetros da relação linear entre `x` e `y` — inclinação `20`, intercepto `5` —, mas imagine que quiséssemos aprendê-los a partir dos dados. Vamos usar gradiente descendente para achar a inclinação e o intercepto que minimizam o erro quadrático médio.

### O gradiente do erro quadrático

Vamos chamar os parâmetros de `theta` (θ) — a convenção deste livro, e de quase todo texto de aprendizado de máquina, para "o vetor de parâmetros do modelo, sejam eles quais forem". O [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html) herda o mesmo nome; do [12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) em diante o vetor passa a se chamar `beta`, seguindo a notação usual de regressão. Aqui, `theta` guarda inclinação e intercepto.

Vamos começar com uma função que determina o gradiente com base no erro de um único ponto de dado:

In [ ]:
from scratch.linear_algebra import Vector

def linear_gradient(x: float, y: float, theta: Vector) -> Vector:
    slope, intercept = theta
    predicted = slope * x + intercept    # a previsão do modelo
    error = (predicted - y)              # o erro é (previsto - real)
    squared_error = error ** 2           # vamos minimizar o erro ao quadrado
    grad = [2 * error * x, 2 * error]    # usando seu gradiente
    return grad

Vale pensar no que esse gradiente significa. Imagine que, para algum `x`, nossa previsão está grande demais. Nesse caso o `error` é positivo. O segundo termo do gradiente, `2 * error`, também é positivo — o que reflete o fato de que pequenos aumentos no intercepto vão deixar a previsão (já grande demais) ainda maior, o que vai fazer o erro quadrático (para esse `x`) aumentar mais ainda.

O primeiro termo do gradiente, `2 * error * x`, tem o mesmo sinal de `x`. De fato, se `x` é positivo, pequenos aumentos na inclinação também vão aumentar a previsão (e, portanto, o erro). Se `x` é negativo, porém, pequenos aumentos na inclinação vão *diminuir* a previsão (e, portanto, o erro).

Esse cálculo foi para um único ponto de dado. Para o conjunto de dados inteiro, olhamos para o **erro quadrático médio**. E o gradiente do erro quadrático médio é simplesmente a média dos gradientes individuais.

### Ajustando theta

Então, aqui está o que vamos fazer:

1. Começar com um valor aleatório para `theta`.
2. Calcular a média dos gradientes.
3. Ajustar `theta` naquela direção.
4. Repetir.

> **🔷 Conceito**
>
> Esta é a receita que o resto do livro vai repetir, trocando só o que entra em cada seta:
>
> **parâmetros aleatórios → gradiente da perda → um passo contra ele → repete**
>
> Troque "inclinação e intercepto" por "os pesos de uma rede neural" e "erro quadrático" por "log-verossimilhança", e é exatamente o que os capítulos 11 a 16 fazem. O laço não muda; só o que ele recebe.

`gradient_step` já foi escrita na [seção 5.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html), e `vector_mean` vem do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html) — a média componente a componente de uma lista de vetores. As duas são o mesmo código que você já escreveu; importamos em vez de reescrever, como a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html) explicou:

In [ ]:
import random
from scratch.gradient_descent import gradient_step
from scratch.linear_algebra import vector_mean

Depois de muitos epochs, deveríamos aprender algo perto dos parâmetros corretos:

In [ ]:
random.seed(0)

# Começa com valores aleatórios para inclinação e intercepto
theta = [random.uniform(-1, 1), random.uniform(-1, 1)]

learning_rate = 0.001

def mse(theta: Vector, dados) -> float:
    slope, intercept = theta
    return sum((slope * x + intercept - y) ** 2 for x, y in dados) / len(dados)

erros = [mse(theta, inputs)]

for epoch in range(5000):
    # Calcula a média dos gradientes
    grad = vector_mean([linear_gradient(x, y, theta) for x, y in inputs])
    # Dá um passo naquela direção
    theta = gradient_step(theta, grad, -learning_rate)
    erros.append(mse(theta, inputs))

# o erro quadrático médio nunca deveria SUBIR de um epoch para o outro
assert all(depois <= antes for antes, depois in zip(erros, erros[1:]))

erros[0], erros[-1]

In [ ]:
slope, intercept = theta
assert 19.9 < slope < 20.1,   "a inclinação deveria ser aproximadamente 20"
assert 4.9 < intercept < 5.1, "o intercepto deveria ser aproximadamente 5"

slope, intercept

O erro quadrático médio cai de mais de `3 × 10⁵` para menos de `10⁻⁷` ao longo de 5.000 epochs — e o `assert` acima confirma, epoch a epoch, que ele nunca sobe. Isso **não** é porque cada passo garante uma perda menor: essa garantia só vale para um passo infinitesimal, e o nosso é finito. O que garante o decréscimo aqui é a mesma lição da [seção 5.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html), agora do lado seguro — `learning_rate = 0.001` é pequeno o bastante frente à curvatura desta perda para que cada passo finito ainda ande na direção certa, em vez de supercorrigir e passar do ponto.

E há uma segunda razão para a busca nunca "se perder": a seção 5.1 avisou que gradiente descendente pode encontrar um mínimo *local* errado, dependendo de onde a busca começa. Aqui isso não é risco — o erro quadrático médio de um modelo linear é uma função **convexa** de `theta`, uma tigela de verdade, sem vales escondidos —, então não importa o ponto de partida aleatório: a busca sempre chega ao mesmo lugar. É por isso que a seção 5.6, adiante, pode trocar a semente e a forma de percorrer os dados sem trocar o destino.

A inclinação e o intercepto encontrados ficam a menos de `0.001` dos valores reais, `20` e `5`.

> **🟩 Exemplo**
>
> Essa mesma conta — inclinação e intercepto que minimizam o erro quadrático — vai aparecer de novo neste livro, por um caminho bem diferente. O [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html) — Regressão Linear Simples — ajusta a mesma coisa com uma **fórmula fechada**: duas médias, uma covariância, uma variância, sem laço nenhum.
>
> Aqui chegamos a esse tipo de resposta por outro caminho — iterando, um pequeno passo de cada vez. Para regressão linear simples, a fórmula fechada é mais rápida e mais exata, e por isso ela existe. Mas ela só existe *porque* o problema é simples o bastante para ter solução fechada; é por isso que o Capítulo 11 pode se dar ao luxo de usá-la, com dados reais em vez de uma reta perfeita. A regressão logística do [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) e as redes neurais do [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) não têm fórmula fechada nenhuma — para elas, o caminho que você acabou de construir aqui é o **único** caminho.

> **💡 Dica — Na prática: `scikit-learn`**
>
> ```python
> from sklearn.linear_model import LinearRegression, SGDRegressor
>
> X = [[x] for x, y in inputs]
> y = [y for x, y in inputs]
>
> # fórmula fechada (mínimos quadrados via álgebra linear)
> modelo_fechado = LinearRegression().fit(X, y)
> modelo_fechado.coef_, modelo_fechado.intercept_
>
> # gradiente descendente estocástico, a mesma ideia da seção 5.6
> modelo_sgd = SGDRegressor(max_iter=1000).fit(X, y)
> modelo_sgd.coef_
> ```
>
> `LinearRegression` não usa gradiente descendente — ela resolve o problema por álgebra linear direta (decomposição em valores singulares), o que é possível porque a regressão linear tem solução fechada. `SGDRegressor` é o que se aproxima do que construímos aqui: ela ajusta os parâmetros passo a passo, e é a opção que o `scikit-learn` recomenda quando o conjunto de dados é grande demais para caber em memória de uma vez, ou quando o modelo — diferente da regressão linear — não tem fórmula fechada. É exatamente esse segundo caso que justifica todo o resto deste capítulo.

## Minibatch e Gradiente Estocástico

> **📌 Nota**
>
> Esta seção corresponde a *Minibatch and Stochastic Gradient Descent*, do capítulo 8 de Grus (2019).

Uma desvantagem da abordagem anterior é que tínhamos que avaliar os gradientes sobre o conjunto de dados **inteiro** antes de dar um único passo de gradiente e atualizar nossos parâmetros. Nesse caso, tudo bem, porque nosso conjunto de dados tinha só 100 pares e o cálculo do gradiente era barato.

Na prática, porém, você vai trabalhar com conjuntos de dados grandes e gradientes caros de calcular. Nesse caso, você vai querer dar passos de gradiente com mais frequência.

### Minibatch

Podemos fazer isso usando uma técnica chamada **gradiente descendente por minibatch**, na qual calculamos o gradiente (e damos um passo de gradiente) com base num "minibatch" amostrado do conjunto de dados maior:

In [ ]:
from typing import TypeVar, List, Iterator

T = TypeVar('T')  # isso nos permite criar funções "genéricas"

def minibatches(dataset: List[T],
                batch_size: int,
                shuffle: bool = True) -> Iterator[List[T]]:
    """Gera minibatches de tamanho `batch_size` a partir do dataset"""
    # Índices de início: 0, batch_size, 2 * batch_size, ...
    batch_starts = [start for start in range(0, len(dataset), batch_size)]

    if shuffle: random.shuffle(batch_starts)  # embaralha os lotes

    for start in batch_starts:
        end = start + batch_size
        yield dataset[start:end]

`TypeVar('T')` só existe para permitir que `minibatches` seja genérica: `dataset` pode ser uma lista de qualquer tipo, e a saída acompanha esse tipo. `minibatches` é o primeiro gerador que este livro usa para valer — se `yield` ainda não é intuitivo, a seção [2.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/05-testes-classes-e-geradores.html) deste livro cobre geradores em detalhe.

`gradient_step` e `linear_gradient` já foram escritas nas seções [5.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html) e [5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html); `vector_mean` vem do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html). Importamos as três, como a [seção 5.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html) explicou:

In [ ]:
import random
from scratch.gradient_descent import gradient_step, linear_gradient
from scratch.linear_algebra import Vector, vector_mean

inputs = [(x, 20 * x + 5) for x in range(-50, 50)]
learning_rate = 0.001

def mse(theta: Vector, dados) -> float:
    slope, intercept = theta
    return sum((slope * x + intercept - y) ** 2 for x, y in dados) / len(dados)

#### O que cada lote realmente contém

Antes de treinar, vale olhar o que cada lote de 20 pontos realmente contém. `theta = [0, 0]` é um bom ponto para medir: nele, o gradiente do conjunto de dados **inteiro** deveria refletir o dataset como um todo.

In [ ]:
theta0 = [0.0, 0.0]

for inicio in range(0, 100, 20):
    lote = inputs[inicio:inicio + 20]
    grad_lote = vector_mean([linear_gradient(x, y, theta0) for x, y in lote])
    print(f"lote x∈[{lote[0][0]}, {lote[-1][0]}]: gradiente = {grad_lote}")

grad_geral = vector_mean([linear_gradient(x, y, theta0) for x, y in inputs])
print(f"dataset inteiro:    gradiente = {grad_geral}")

> **⚠️ Atenção — Um `shuffle` que não embaralha**
>
> `minibatches` embaralha `batch_starts` — os **índices de início** dos lotes —, não os pontos do dataset. Como `inputs` está ordenado por `x`, cada lote é sempre a mesma fatia contígua: o primeiro é sempre "x de −50 a −31", o último é sempre "x de 30 a 49". Embaralhar a *ordem* em que essas cinco fatias fixas são processadas não muda o que cada fatia *é*.
>
> E o que cada fatia é, é uma amostra sistematicamente torta — não ruidosa, **torta**. Acima, o gradiente do dataset inteiro tem um termo de intercepto quase nulo (`10`), porque os erros positivos e negativos se cancelam quando você olha para todo `x`. Mas o termo de intercepto de cada lote isolado varia de `+1610` a `-1590` — o **sinal inverte** conforme a fatia está do lado negativo ou positivo de `x`. Cada lote empurra `theta` na direção que corrige *aquela fatia*, não na direção que corrige o dataset inteiro. Embaralhar a ordem dos cinco não desfaz o viés de nenhum deles.
>
> O conserto tem uma linha: embaralhar o **dataset**, de novo a cada epoch, antes de gerar os lotes — não os índices de início.

In [ ]:
def treina_minibatch(embaralha_dataset: bool, seed: int, n_epochs: int = 1000):
    random.seed(seed)
    theta = [random.uniform(-1, 1), random.uniform(-1, 1)]
    dataset = inputs[:]
    erros = [mse(theta, inputs)]
    for epoch in range(1, n_epochs + 1):
        if embaralha_dataset:
            random.shuffle(dataset)  # <-- o conserto de uma linha
        for batch in minibatches(dataset, batch_size=20, shuffle=not embaralha_dataset):
            grad = vector_mean([linear_gradient(x, y, theta) for x, y in batch])
            theta = gradient_step(theta, grad, -learning_rate)
        if epoch % 20 == 0:
            erros.append(mse(theta, inputs))
    return theta, erros

theta_enviesado, erros_enviesado = treina_minibatch(embaralha_dataset=False, seed=1)
theta_corrigido, erros_corrigido = treina_minibatch(embaralha_dataset=True, seed=1)

def subidas(erros):
    """Quantas vezes o erro sobe de um checkpoint para o outro."""
    return sum(1 for antes, depois in zip(erros, erros[1:]) if depois > antes)

subidas(erros_enviesado), subidas(erros_corrigido), len(erros_enviesado) - 1

Em 50 checkpoints ao longo de 1.000 epochs, a versão original sobe 18 vezes; a corrigida, 7. Não zera — um lote de 20 pontos, mesmo sorteado direito, ainda é uma amostra, e amostras têm variância —, mas a maior fonte de ruído, o viés sistemático das fatias fixas, desaparece. No epoch 100, por exemplo, o erro da versão original está em `55,8`; o da corrigida, em `2,6`. Daqui em diante, o resto desta seção usa a versão corrigida — embaralhando o dataset a cada epoch.

### Gradiente estocástico

Outra variação é o **gradiente descendente estocástico**, no qual você dá passos de gradiente com base em um único exemplo de treino por vez:

In [ ]:
random.seed(2)
theta = [random.uniform(-1, 1), random.uniform(-1, 1)]

for epoch in range(100):
    for x, y in inputs:
        grad = linear_gradient(x, y, theta)
        theta = gradient_step(theta, grad, -learning_rate)

slope, intercept = theta
assert 19.9 < slope < 20.1,   "a inclinação deveria ser aproximadamente 20"
assert 4.9 < intercept < 5.1, "o intercepto deveria ser aproximadamente 5"

slope, intercept

Em 100 epochs — um décimo do minibatch, um cinquentavo do lote inteiro —, o estocástico também chega perto de `[20, 5]`.

### Comparando os três

Só que "100 epochs" e "5.000 epochs" não medem a mesma coisa. Um epoch de lote inteiro dá 1 passo de gradiente; um epoch de minibatch (lotes de 20) dá 5; um epoch estocástico dá 100 — um por ponto. Comparar pelo número de epochs é comparar corridas com orçamentos de trabalho diferentes.

> **📌 Nota — Quantas vezes `gradient_step` foi chamado**
>
> Lote inteiro: `5.000 epochs × 1 passo = 5.000` chamadas. Minibatch: `1.000 epochs × 5 passos = 5.000` chamadas — o mesmo total, apesar de dez vezes menos epochs. Estocástico: `100 epochs × 100 passos = 10.000` chamadas — o dobro dos outros dois, apesar de ser o que "venceu" em epochs.

A comparação justa é pelo número de chamadas a `gradient_step`, não pelo número de epochs:

In [ ]:
# Figura: Erro quadrático médio contra o número de chamadas a gradient_step, para lote inteiro, minibatch (corrigido) e estocástico (escala log-log) — repare no serrilhado da curva estocástica
# Reexecuta os três laços já mostrados nesta seção, desta vez registrando o
# erro a cada passo (não a cada epoch), para comparar por número de chamadas
# a gradient_step em vez de epochs. Código igual ao das seções anteriores.
# lote inteiro: 5.000 epochs = 5.000 passos
random.seed(0)
theta = [random.uniform(-1, 1), random.uniform(-1, 1)]
passos_lote, erros_lote = [], []
passo = 0
for epoch in range(5000):
    grad = vector_mean([linear_gradient(x, y, theta) for x, y in inputs])
    theta = gradient_step(theta, grad, -learning_rate)
    passo += 1
    if passo % 10 == 0:
        passos_lote.append(passo)
        erros_lote.append(mse(theta, inputs))

# minibatch corrigido: 1.000 epochs x 5 passos = 5.000 passos
random.seed(1)
theta = [random.uniform(-1, 1), random.uniform(-1, 1)]
dataset = inputs[:]
passos_mb, erros_mb = [], []
passo = 0
for epoch in range(1000):
    random.shuffle(dataset)
    for batch in minibatches(dataset, batch_size=20, shuffle=False):
        grad = vector_mean([linear_gradient(x, y, theta) for x, y in batch])
        theta = gradient_step(theta, grad, -learning_rate)
        passo += 1
        if passo % 10 == 0:
            passos_mb.append(passo)
            erros_mb.append(mse(theta, inputs))

# estocastico: 100 epochs x 100 passos = 10.000 passos
random.seed(2)
theta = [random.uniform(-1, 1), random.uniform(-1, 1)]
passos_sgd, erros_sgd = [], []
passo = 0
for epoch in range(100):
    for x, y in inputs:
        grad = linear_gradient(x, y, theta)
        theta = gradient_step(theta, grad, -learning_rate)
        passo += 1
        if passo % 10 == 0:
            passos_sgd.append(passo)
            erros_sgd.append(mse(theta, inputs))

from matplotlib import pyplot as plt

plt.plot(passos_lote, erros_lote, label="lote inteiro (5.000 epochs)",
         linestyle="--", linewidth=2.5)
plt.plot(passos_mb, erros_mb, label="minibatch (1.000 epochs)",
         linestyle="-", linewidth=1.3)
plt.plot(passos_sgd, erros_sgd, label="estocástico (100 epochs)",
         linestyle="-", linewidth=1, alpha=0.7)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("número de chamadas a gradient_step")
plt.ylabel("erro quadrático médio")
plt.legend()
plt.show()

Repare primeiro nas curvas tracejada (lote inteiro) e fina (minibatch): a partir de umas poucas dezenas de chamadas, elas praticamente se sobrepõem, e a tracejada corre por baixo da fina pelo resto do gráfico. Isso não é um defeito da figura — é o próprio argumento desta seção: com o mesmo orçamento de chamadas a `gradient_step`, as duas chegam à mesma precisão.

O que salta aos olhos é a linha verde: ela não desce, ela **serrilha**. Só o epoch 1 dispara acima de `10¹⁵` — para `6,9 × 10¹⁸`, bem acima do resto do gráfico —; os picos dos epochs seguintes, embora recorrentes, ficam sistematicamente abaixo dessa faixa, o maior deles (epoch 2) em `5,7 × 10¹⁴`. Não é ruído de plotagem — é `gradient_step` divergindo, ponto a ponto, exatamente como a caixa da [seção 5.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html) descreveu para um `v` inteiro.

`inputs` está ordenado por `x`, e o laço estocástico processa os pontos nessa ordem — começando sempre por `x = -50`. No início do primeiro epoch, com `theta` ainda longe do valor certo, o gradiente naquele único ponto já é enorme:

In [ ]:
random.seed(2)
theta_demo = [random.uniform(-1, 1), random.uniform(-1, 1)]
for x, y in inputs[:4]:
    grad = linear_gradient(x, y, theta_demo)
    theta_demo = gradient_step(theta_demo, grad, -learning_rate)
    print(f"x = {x:4d}  theta = {theta_demo}")

`theta` pula de perto de `[0.9, 0.9]` para `[95.9, -1.0]` — passa direto pelo `20` certo —, e o próximo ponto (`x = -49`), vendo um erro ainda maior, supercorrige para o lado oposto, `[-269, 6.4]`. É a mesma cascata de supercorreção da seção 5.4, só que provocada por um único ponto extremo em vez de um `step_size` grande demais: o `learning_rate = 0.001`, perfeitamente estável para o gradiente **médio** do dataset inteiro, é grande demais para a curvatura de um ponto isolado com `x` perto de `±50`.

A cascata se autocorrige — depois de processar os 100 pontos do epoch, com erros de sinais que se cancelam parcialmente, `theta` volta para perto do valor certo —, e o pico de cada epoch é menor que o do anterior: de `6,9 × 10¹⁸` no epoch 1 para `1,0 × 10¹¹` no epoch 100. Mas o padrão nunca desaparece de todo: a cada novo epoch, o primeiro ponto (`x = -50`) provoca um novo solavanco, cada vez menor, e é isso que o serrilhado da figura mostra do início ao fim.

Medido só nas fronteiras de epoch — onde a cascata já se autocorrigiu — o quadro é mais parecido com o que se esperaria: no mesmo orçamento de `5.000` chamadas (fronteira do epoch 50), o estocástico está em `0,34`, contra `4,1 × 10⁻⁸` do lote inteiro e `3,8 × 10⁻⁸` do minibatch — sete ordens de grandeza pior, e isso sem contar os picos. Mesmo esgotando as `10.000` chamadas que de fato usou, chega só a `4,2 × 10⁻³`, ainda muito atrás da precisão que os outros dois atingem com metade do orçamento — e pagando o preço de uma trajetória bem mais instável para chegar lá.

Isso não quer dizer que o estocástico seja inútil: ele chega aos parâmetros certos processando um ponto de cada vez, com cada atualização individual muito mais barata que a de um lote de 20 ou de 100 pontos. Mas "barata por chamada" não é "estável" — e é exatamente esse tipo de comportamento que se evita, na prática, reescalonando os atributos antes de treinar (para que nenhuma dimensão tenha curvatura muito maior que as outras — o assunto da seção [7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html)) ou usando um passo bem menor para atualizações de um ponto só.

> **🔷 Conceito**
>
> O tamanho do lote é um controle de compromisso, não uma escolha "certa" universal. Lotes maiores dão um gradiente mais fiel ao dataset inteiro — mais estável, mas mais caro por passo, e menos passos pelo mesmo custo de "passar pelos dados uma vez". Lotes menores dão mais passos, cada um mais barato — mas cada um enxerga menos dado, e no limite (um ponto só) pode enxergar um ponto atípico o bastante para que o mesmo `learning_rate`, seguro para a média do dataset, vire grande demais para aquele ponto isolado.

Ao longo do livro, vamos experimentar para achar tamanhos de lote e de passo adequados a cada problema.

> **📌 Nota**
>
> A terminologia para as várias variantes de gradiente descendente não é uniforme. A abordagem "calcule o gradiente para o conjunto de dados inteiro" costuma ser chamada de **gradiente descendente em lote** (*batch gradient descent*), e algumas pessoas dizem **gradiente descendente estocástico** ao se referir à versão por minibatch (da qual a versão ponto a ponto é um caso especial).

O laço que você escreveu nesta seção — dividir os dados em lotes, calcular o gradiente médio de cada lote, chamar `gradient_step`, repetir — não muda mais neste livro. O que muda, capítulo a capítulo, é só o gradiente que entra nele. Nos capítulos [11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html) e [12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html), regressão linear simples e múltipla, é o gradiente do erro quadrático — o mesmo `linear_gradient` que você acabou de usar aqui, só que com mais coeficientes. No [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html), regressão logística, é o gradiente da log-verossimilhança — outra função, mesma receita. Nos capítulos [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html), redes neurais e deep learning, é o gradiente calculado por retropropagação, uma rede inteira de derivadas encadeadas em vez de duas. Você não vai reescrever `gradient_step` nenhuma vez a mais — vai só trocar o que ele recebe.

> **💡 Dica — Na prática: `batch_size` é um hiperparâmetro de verdade**
>
> Todo framework de deep learning expõe exatamente essa escolha, com esse nome:
>
> ```python
> # PyTorch
> from torch.utils.data import DataLoader
> DataLoader(dataset, batch_size=32, shuffle=True)
>
> # Keras / TensorFlow
> modelo.fit(X_treino, y_treino, batch_size=32, epochs=10)
> ```
>
> Hoje em dia, quando alguém diz "treinei com SGD", quase sempre quer dizer gradiente descendente por minibatch — a ressalva de terminologia acima não é só uma curiosidade histórica, é como a maioria dos textos e das APIs usa o termo. `batch_size = 1` (o estocástico puro, ponto a ponto) é raro em produção, e a comparação acima mostra por quê: ele gasta mais chamadas para chegar à mesma precisão que lote inteiro ou minibatch. Se você não estivesse fazendo a sua álgebra linear do zero, a diferença seria ainda maior: bibliotecas como NumPy calculam o gradiente de um lote inteiro numa única operação vetorizada, em vez de somar as contribuições de cada ponto uma a uma em Python puro — outro motivo, além da precisão por chamada, pelo qual minibatches maiores costumam ganhar de exemplos únicos na prática.
>
> Os otimizadores que você vai encontrar nos capítulos sobre [redes neurais](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [deep learning](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) — Adam, RMSProp, SGD com momentum — são todos construídos **em cima** de gradiente descendente por minibatch, não no lugar dele: eles mudam como o tamanho do passo se adapta a cada minibatch, mas o laço externo — separar os dados em lotes, calcular o gradiente médio de cada lote, chamar o equivalente de `gradient_step` — é o mesmo que você acabou de escrever aqui.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 8 de Grus (2019) sugere:

- Continuar lendo — o próprio autor observa que o livro vai usar gradiente descendente para resolver problemas até o fim, e de fato: é o que os capítulos 11, 12, 13, 15 e 16 deste livro fazem.
- *[Active Calculus 1.0](https://activecalculus.org/)*, de Matthew Boelkins, David Austin e Steven Schlicker (Grand Valley State University Libraries), disponível gratuitamente, para quem quiser revisar cálculo.
- O [texto de Sebastian Ruder](https://www.ruder.io/optimizing-gradient-descent/) comparando gradiente descendente e suas muitas variantes — momentum, RMSProp, Adam — que reaparecem nos capítulos sobre redes neurais.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.